[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SubSurfObs/observatory_notebooks/blob/main/01_vw_waveform_analysis/index.ipynb)

This notebook fetches the most recent 24 hours of waveform data from the
[University of Melbourne Seismic Network (VW)](https://subsurface.science.unimelb.edu.au)
via the FDSN web services and plots a multi-station waveform record section.

Data are fetched live — the output on this page is a snapshot; click the Colab badge above to run with current data.

In [ ]:
# Install dependencies when running on Colab (already present in local env)
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'obspy'], check=True)

In [ ]:
from obspy import UTCDateTime
from obspy.clients.fdsn import Client
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

FDSN_HOST = 'https://subsurface.science.unimelb.edu.au'
NETWORK   = 'VW'
CHANNEL   = 'HHZ'   # high-gain vertical
HOURS     = 24

client = Client(base_url=FDSN_HOST)
print(f'Connected to {FDSN_HOST}')

## Fetch station list

Query the station service to discover which VW stations are currently active.

In [ ]:
inv = client.get_stations(
    network=NETWORK,
    channel=CHANNEL,
    level='channel',
)

stations = [
    sta.code
    for net in inv
    for sta in net
]
print(f'Active {NETWORK} stations with {CHANNEL}: {stations}')

## Fetch waveforms

Fetch the most recent 24 hours of data for all active stations.

In [ ]:
t_end   = UTCDateTime()          # now
t_start = t_end - HOURS * 3600

print(f'Fetching {t_start.date} {str(t_start.time)[:5]} → {t_end.date} {str(t_end.time)[:5]} UTC')

st = client.get_waveforms(
    network=NETWORK,
    station='*',
    location='*',
    channel=CHANNEL,
    starttime=t_start,
    endtime=t_end,
)

st.merge(fill_value='interpolate')
st.detrend('demean')
print(st)

## Record section

Plot all traces offset by station, normalised to a common amplitude scale.

In [ ]:
NAVY = '#000F46'
PINK = '#E5007D'

fig, ax = plt.subplots(figsize=(12, max(4, len(st) * 1.2)))

for i, tr in enumerate(sorted(st, key=lambda t: t.stats.station)):
    times = tr.times('matplotlib')
    data  = tr.data.astype(float)
    # Normalise to ±0.4 of inter-trace spacing
    amp   = data / (data.std() + 1e-12) * 0.35
    ax.plot(times, amp + i, color=NAVY, linewidth=0.5, alpha=0.85)
    ax.text(times[0], i + 0.4, tr.stats.station,
            fontsize=8, color=PINK, va='bottom')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.xaxis.set_major_locator(mdates.HourLocator(interval=4))
fig.autofmt_xdate()

ax.set_yticks([])
ax.set_xlabel(f'UTC time ({t_start.date})', fontsize=10)
ax.set_title(f'{NETWORK} network — {CHANNEL} — {HOURS} h record section', fontsize=12)
ax.spines[['top','right','left']].set_visible(False)

plt.tight_layout()
plt.show()

---

**Data source:** University of Melbourne Seismic Network (VW),
DOI [10.7914/8csc-8z27](https://doi.org/10.7914/8csc-8z27).  
**Licence:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).  
**Access:** `https://subsurface.science.unimelb.edu.au/fdsnws/`